# HISCA 一键启动

本笔记本负责准备环境和启动现有预测网页，不改变模型。

**使用方法：** 按顺序运行下面四个代码单元，或选择菜单 **Run → Run All Cells**。首次安装需要联网。启动完成后点击链接使用，也可以在本笔记本内操作网页。

完整本地包已经包含数据、特征、权重和 tokenizer。GitHub 代码版不包含大文件，需要先向维护者取得同一套部署文件。

支持 Python 3.9–3.12；本次验证使用 3.9.23。本地运行，不是 Colab/Binder 在线托管版本。

## 第一步：确认位置和数据

如果使用完整本地包，DATA_ROOT 保持空字符串即可。若部署文件在其他位置，请改为包含 data、features、ckpt 三个文件夹的上级目录。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "hisca_quickstart.py").is_file():
    raise RuntimeError("请将 Notebook 放在 HISCA 项目根目录并从该目录打开。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from hisca_quickstart import check_files, ensure_dependencies, Platform
DATA_ROOT = ""  # 完整本地包不需要修改；也可填自己的部署包目录
paths = check_files(DATA_ROOT or None)
print("文件检查通过。当前 Python:", sys.executable)
for key, value in paths.items():
    print(key, ":", value)


## 第二步：准备推理依赖

依赖安装到当前 Notebook 内核对应的 Python 中。如果已安装兼容版本，会直接跳过。首次缺少 PyTorch 时默认安装 CPU 版本；现有匹配的 CUDA 版本会保留。

首次安装后若提示版本或导入异常，请重启内核再运行全部单元。

In [ ]:
ensure_dependencies()


## 第三步：启动平台

首次加载需要等待。启动助手会自动选择空闲的本地端口，并把日志保存在 .runtime/notebook_server.log。

重复运行此单元会复用本 Notebook 已启动的服务，不重复加载模型。

In [ ]:
requested_root = DATA_ROOT or None
if "hisca" in globals() and hisca.data_root != requested_root:
    hisca.stop()
    del hisca
if "hisca" not in globals():
    hisca = Platform(requested_root)
url = hisca.start()
print("浏览器访问:", url)


## 第四步：使用预测网页

配对预测：选择已有 ncRNA 与药物，点击 Predict。候选排序：进入 Drug → top ncRNAs，选择药物、类型和数量后点击 Rank。

若内嵌页面未显示，点击下方链接在普通浏览器打开。请保持 Notebook 内核运行，关闭内核会停止它启动的网页服务。

In [ ]:
from IPython.display import display, HTML, IFrame
display(HTML(f'<p><a href="{url}" target="_blank">打开 HISCA 预测平台</a></p>'))
display(IFrame(src=url, width="100%", height=850))


## 停止与排错

不再使用时，新建一个单元运行 hisca.stop()，或者关闭本 Notebook 的内核。

如启动失败，可以运行 print(hisca.tail()) 查看日志。停止操作仅针对本 Notebook 创建的进程，不会关闭其他人的本地服务。

本网页不接受任意新序列或自定义 SMILES。分数与交互热图用于研究探索，不能替代实验验证。